# Chạy split test MỘT LẦN trên Colab (issue #11)

Cấu hình đã chốt trên dev nằm trong `evaluation/frozen_config.json` (model M = e5-base, text v1, P1).
Notebook này chỉ encode 20 câu test; vector node dùng lại đúng bản đã chấm trên dev (`--require-cache`).

1. Runtime → Change runtime type → **T4 GPU**.
2. Chạy ô 1–3. Ô upload: chọn `colab_test_bundle.zip` trong thư mục dự án.
3. Nếu ô 3 báo đã cài lại thư viện: **Runtime → Restart session**, rồi chạy lại ô 3 (phải báo khớp).
4. Chạy ô 4 (test) và ô 5 (tải `colab_test_results.zip`), đưa file về thư mục gốc dự án và báo lại.

Không chạy ô 4 nhiều lần: mỗi lần chạy test đều được đếm trong `test_runs.json` và ghi vào báo cáo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
import os, zipfile
from google.colab import files
os.chdir('/content')
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
bundle = '/content/colab_test_bundle.zip'
if not os.path.exists(bundle):
    files.upload(target_dir='/content')  # chọn colab_test_bundle.zip
zipfile.ZipFile(bundle).extractall(WORK)
os.chdir(WORK)
!ls legal_kg evaluation

In [ ]:
# Vector node chỉ dùng lại được khi thư viện, GPU và revision giống hệt lúc encode dev.
import os, json, subprocess, sys, importlib.metadata as m
WORK = '/content/work'
os.chdir(WORK)
manifest = json.load(open('output/kg/embedding_benchmark/e5-base/manifest.json', encoding='utf-8'))
want = manifest['config']['versions']
have = {p: m.version(p) for p in want}
print('cần :', want)
print('đang:', have)
if have != want:
    torch_version, _, cuda = want['torch'].partition('+')
    if have['torch'] != want['torch']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'torch=={torch_version}',
                        '--index-url', f'https://download.pytorch.org/whl/{cuda}'], check=True)
    others = [f'{p}=={v}' for p, v in want.items() if p != 'torch' and have[p] != v]
    if others:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *others, 'huggingface-hub>=0.34,<2'], check=True)
    print('ĐÃ CÀI LẠI → Runtime → Restart session, rồi chạy lại ô này.')
else:
    print('Khớp môi trường dev.')

In [ ]:
import os, json, subprocess, sys
WORK = '/content/work'
os.chdir(WORK)
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
frozen = json.load(open('evaluation/frozen_config.json', encoding='utf-8'))
CORPORA = [('output/kg/embedding_text/node_texts.jsonl', 'output/kg/embedding_benchmark'),
           ('output/kg/embedding_text_v2/node_texts.jsonl', 'output/kg/embedding_benchmark/text-v2')]
for model in frozen['models']:
    for corpus, output in CORPORA:
        config = json.load(open(f'{output}/{model}/manifest.json', encoding='utf-8'))['config']
        command = [sys.executable, '-m', 'legal_kg.embedding_benchmark', '--model', model, '--device', 'cuda',
                   '--threads', str(config['threads']), '--batch-size', str(config['batch_size']),
                   '--corpus', corpus, '--output', output, '--questions', 'evaluation/test.jsonl',
                   '--frozen-config', 'evaluation/frozen_config.json', '--require-cache']
        print('$', ' '.join(command[2:]), flush=True)
        result = subprocess.run(command, capture_output=True, text=True)
        print(result.stdout[-1200:])
        if result.returncode != 0:
            print(result.stderr[-3000:])
            raise RuntimeError(f'Failed: {model} {output}')
print('ALL DONE')

In [ ]:
import os
os.chdir('/content/work')
!rm -f colab_test_results.zip && zip -qr colab_test_results.zip output/kg/embedding_benchmark/test output/kg/embedding_benchmark/text-v2/test
!ls -lh colab_test_results.zip
from google.colab import files
files.download('/content/work/colab_test_results.zip')